# <Question in one line>
Window: last `<N>` days up to `<end date, exclusive>`. Run with the read-only role from `00_connection.ipynb`.

In [ ]:
# Connection only. Every value comes from an environment variable; nothing secret is stored here.
import os

import pandas as pd
import snowflake.connector

key_file = os.environ.get("SNOWFLAKE_PRIVATE_KEY_FILE")
auth = ({"authenticator": "SNOWFLAKE_JWT", "private_key_file": key_file} if key_file
        else {"authenticator": os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser")})
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    role=os.environ["SNOWFLAKE_ROLE"],            # a read-only role
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],  # X-Small is enough for system views
    session_parameters={"TIMEZONE": "UTC", "QUERY_TAG": "analysis:<topic>"},
    **auth,
)


def q(sql, params=None):
    """Run one read-only statement and return a DataFrame (SHOW output too)."""
    cur = conn.cursor()
    try:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    finally:
        cur.close()

### 0. Can this role see the views the answer needs?
- **What:** counts last-7-day rows in each view the next queries read.
- **Reads:** <VIEW_1>, <VIEW_2> (no join).
- **Look for:** an error or 0 rows = not visible to this role (name the SNOWFLAKE database role needed), not "nothing found". Not counted in the 1-4 queries.

In [ ]:
q("""
SELECT '<VIEW_1>' AS src, COUNT(*) AS rows_7d
FROM SNOWFLAKE.ACCOUNT_USAGE.<VIEW_1>
WHERE <time_column> >= DATEADD(day, -7, CURRENT_TIMESTAMP())
""")

### 1. <Query title>
- **What:** <what the query computes, per what grain, over which window>.
- **Reads:** <views> (<grain of each>); join on <key>.
- **Look for:** <columns that answer the question, units, normal vs bad values, pitfalls>.

In [ ]:
q("""
WITH p AS (SELECT 30 AS days, CURRENT_DATE() AS end_date)
SELECT <columns>
FROM SNOWFLAKE.ACCOUNT_USAGE.<VIEW_1> v, p
WHERE v.<time_column> >= DATEADD(day, -p.days, p.end_date) AND v.<time_column> < p.end_date
GROUP BY <keys>
ORDER BY <measure> DESC
LIMIT 1000
""")

## Findings
- Confirmed: <statement> (query N).
- Not verified: <statement> - next query: <one line>.